# Inverse FX Predictor — Gradio test app (Phase 1, Log-Mel + Transformer)

Pick a **test** pair (dry + wet audio from songs the model never saw), run the model, and compare the
**prediction** with the **actual** effects underneath. A second tab lets you upload your own dry/wet files.

**Kaggle settings:** Internet **ON** · Add-ons → Secrets → `HF_TOKEN` (read access is enough) ·
Accelerator: *None* is fine (2.2 M-parameter model, CPU inference < 1 s). GPU also works.

**Run:** *Run All*, then open the **public gradio.live link** printed by the last-but-one cell
(Kaggle cannot show the app inline). The link lives as long as this session; run the last cell to stop it.

## 1. Setup

In [ ]:
import os, sys, json, time, shutil, importlib, subprocess
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

if importlib.util.find_spec("gradio") is None or int(importlib.import_module("gradio").__version__.split(".")[0]) < 5:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=5.9"])
for mod, spec in {"datasets": "datasets>=2.19", "soundfile": "soundfile", "librosa": "librosa"}.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import numpy as np, pandas as pd, torch, soundfile as sf, librosa, gradio as gr
from matplotlib.figure import Figure
print(f"torch {torch.__version__} | gradio {gr.__version__} | Python {sys.version.split()[0]}")

## 2. Configuration

In [ ]:
MODEL_REPO = "APINAJA/inverse-fx-phase1-logmel-transformer-v2"
DATASET_ID = "APINAJA/inverse-fx-phase1-effect-type"
DATASET_REVISION = "b95597add82924cbe82b2e7015c057ffa2a51fba"   # same dataset version the model was trained on
PAIRS_PER_COMBINATION = 2        # 7 combinations x 2 = 14 test pairs
MAX_PAIRS_PER_SONG = 2           # spread the demo pairs over several songs
MAX_ROWS_TO_SCAN = 462           # whole test split at most (streamed, never fully downloaded)
REBUILD_DEMO = False             # True = pick the demo pairs again

WORK = "/kaggle/working/fx_demo" if os.path.exists("/kaggle/working") else os.path.abspath("fx_demo")
MODEL_DIR, DEMO_DIR = os.path.join(WORK, "model_repo"), os.path.join(WORK, "demo_data")
os.makedirs(DEMO_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| work dir:", WORK)

## 3. Hugging Face login
Token from the Kaggle secret `HF_TOKEN` (never printed).

In [ ]:
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
_tok = _tok or os.environ.get("HF_TOKEN")
if _tok:
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import HfApi
    print("Logged in to Hugging Face as:", HfApi(token=_tok).whoami().get("name"))
else:
    print("WARNING: no HF_TOKEN — a private model repo cannot be downloaded.")
del _tok

## 4. Load the trained model
Downloads `final/` (weights + config), `fxp1/` (the exact model code used in training) and `results/` (Kaggle test predictions, used to verify this app).

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(MODEL_REPO, repo_type="model", local_dir=MODEL_DIR, token=os.environ.get("HF_TOKEN"),
                  allow_patterns=["fxp1/*", "final/*", "results/test_predictions.csv", "results/test_metrics.json"])
for m in [m for m in sys.modules if m == "fxp1" or m.startswith("fxp1.")]:
    del sys.modules[m]                       # re-runs pick up the downloaded code
sys.path.insert(0, MODEL_DIR)

from fxp1.model import build_model, count_parameters
from fxp1.audio import AudioPreprocessor
from fxp1.metrics import predict_valid_vectors
from fxp1.common import EFFECTS, COMBINATIONS, VECTOR_TO_COMBINATION

CFG = json.load(open(os.path.join(MODEL_DIR, "final", "config.json")))["experiment_config"]
model = build_model(CFG)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "final", "model.pt"), map_location="cpu"))
model.to(DEVICE).eval()
PRE = AudioPreprocessor(CFG)
THRESHOLD = float(CFG["THRESHOLD"])
SR, N_SAMPLES = PRE.sample_rate, PRE.num_samples

state = json.load(open(os.path.join(MODEL_DIR, "final", "training_state.json")))
tm = json.load(open(os.path.join(MODEL_DIR, "final", "test_metrics.json")))
print(f"Model loaded: {count_parameters(model):,} params | best epoch {state['best_epoch']} | threshold {THRESHOLD}")
print(f"Kaggle test results: macro F1 {tm['macro_f1']:.3f} | exact match {tm['exact_match_accuracy']:.3f} | "
      + " | ".join(f"{e} F1 {tm[e + '/f1']:.3f}" for e in EFFECTS))

## 5. Inference helpers
Same preprocessing as training (`fxp1.audio.AudioPreprocessor`: mono, 44.1 kHz, first 10 s / zero-pad, float32, no normalisation). Ground-truth metadata is never passed to the model.

In [ ]:
@torch.no_grad()
def predict(dry_wave, wet_wave):
    d = torch.from_numpy(np.ascontiguousarray(dry_wave))[None].to(DEVICE)
    w = torch.from_numpy(np.ascontiguousarray(wet_wave))[None].to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        logits = model(d, w)                               # [1, 3]  (fp16 on GPU = same as Kaggle evaluation)
    probs = torch.sigmoid(logits.float())[0].cpu().numpy()
    pv = predict_valid_vectors(probs[None], THRESHOLD)
    combo = COMBINATIONS[VECTOR_TO_COMBINATION[tuple(int(v) for v in pv["valid"][0])]][0]
    return {"probs": probs, "pred": pv["raw"][0].astype(int), "combo": combo, "all_zero": bool(pv["all_zero"][0])}


def load_file_for_model(path):
    """Decode exactly like training (soundfile) when possible; librosa fallback for mp3 etc."""
    try:
        return PRE({"path": path, "bytes": None})
    except Exception:
        arr, sr = librosa.load(path, sr=None, mono=False)
        return PRE({"array": np.atleast_2d(arr).astype(np.float32), "sampling_rate": sr})


def spectrogram_figure(dry_wave, wet_wave, title=""):
    with torch.no_grad():
        lm = model.logmel.raw_log_mel(torch.from_numpy(np.stack([dry_wave, wet_wave])).to(DEVICE)).cpu().numpy()
    diff = lm[1] - lm[0]
    fig = Figure(figsize=(15, 3.3), dpi=90)
    axes = fig.subplots(1, 3, sharey=True)
    lo, hi = np.percentile(lm, 5), lm.max()
    ext = [0, CFG["AUDIO_LENGTH_SECONDS"], 0, CFG["N_MELS"]]
    for ax, m, t in zip(axes[:2], lm, ["Dry log-mel", "Wet log-mel"]):
        ax.imshow(m, origin="lower", aspect="auto", cmap="magma", vmin=lo, vmax=hi, extent=ext)
        ax.set_title(t, loc="left", fontsize=10)
    lim = float(np.percentile(np.abs(diff), 99)) or 1.0
    im = axes[2].imshow(diff, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
    axes[2].set_title("Wet − Dry (red = louder in wet, blue = quieter)", loc="left", fontsize=10)
    fig.colorbar(im, ax=axes[2], fraction=0.04, pad=0.02, label="dB-like")
    for ax in axes:
        ax.set_xlabel("Time (s)", fontsize=9); ax.tick_params(labelsize=8)
    axes[0].set_ylabel("Mel bin", fontsize=9)
    if title:
        fig.suptitle(title, x=0.01, ha="left", fontsize=10)
    fig.tight_layout()
    return fig


def _fmt_params(p):
    if not p:
        return "—"
    try:
        d = json.loads(p) if isinstance(p, str) else p
    except Exception:
        return str(p)
    units = {"frequency": " Hz", "gain": " dB", "threshold": " dB", "attack": " ms", "release": " ms",
             "makeup_gain": " dB"}
    return ", ".join(f"{k} {v:g}{units.get(k, '')}" if isinstance(v, (int, float)) else f"{k} {v}" for k, v in d.items())


def result_markdown(res, truth=None):
    probs, pred = res["probs"], res["pred"]
    out = ["### Model prediction", f"**Predicted combination: {res['combo']}**"]
    if res["all_zero"]:
        out.append("_No effect passed the threshold; the most probable one is shown._")
    out += ["", "| Effect | Probability | Predicted |", "|---|---|---|"]
    out += [f"| {e} | {probs[k]:.3f} | {'present' if pred[k] else 'absent'} |" for k, e in enumerate(EFFECTS)]
    if truth is None:
        return "\n".join(out)
    act = truth["labels"]
    n_ok = int(sum(int(pred[k]) == int(act[k]) for k in range(3)))
    exact = n_ok == 3
    out += ["", "---", "### Actual (ground truth)", f"**True combination: {truth['combination']}**", "",
            "| Effect | Actual | Predicted | Result |", "|---|---|---|---|"]
    out += [f"| {e} | {'present' if act[k] else 'absent'} | {'present' if pred[k] else 'absent'} | "
            f"{'✅ correct' if int(pred[k]) == int(act[k]) else '❌ wrong'} |" for k, e in enumerate(EFFECTS)]
    out += ["", f"**Exact match: {'✅ yes' if exact else '❌ no'}** ({n_ok}/3 effects correct)", "",
            "**Settings used to create this wet audio** (shown only for analysis — never given to the model):",
            f"- EQ: {_fmt_params(truth.get('eq_params'))}",
            f"- Compressor: {_fmt_params(truth.get('compressor_params'))}",
            f"- Reverb: {_fmt_params(truth.get('reverb_params'))}"]
    if truth.get("effect_change_db"):
        out.append(f"- Effect change (dataset metric): {truth['effect_change_db']}")
    if truth.get("low_effect_change") not in (None, "", "[]"):
        out.append(f"- ⚠️ Flagged as a *low* (subtle) effect change in the dataset: {truth['low_effect_change']}")
    out.append(f"\n<sub>{truth['sample_id']} · {truth['source_category']} · {truth['source_song']}</sub>")
    return "\n".join(out)


def probs_for_label(res):
    return {e: float(res["probs"][k]) for k, e in enumerate(EFFECTS)}

print("helpers ready")

## 6. Pick the test pairs
Streams the **test** split (no full download) and keeps 2 pairs per combination, spread over Bass / Drums / Mixture and several songs. Saved to `demo_data/` so re-runs are instant.

In [ ]:
CATS = ["Bass", "Drums", "Mixture"]
DEMO_JSON = os.path.join(DEMO_DIR, "demo.json")


def _labels(row):
    return (int(row["eq_label"]), int(row["compressor_label"]), int(row["reverb_label"]))


def select_rows(rows, per_combo=PAIRS_PER_COMBINATION, max_per_song=MAX_PAIRS_PER_SONG, max_rows=MAX_ROWS_TO_SCAN):
    """Deterministic: slot s of combination c prefers category CATS[(c + s) % 3]; <= max_per_song picks per song."""
    picked = {c: [] for c in range(7)}
    backup = {c: [] for c in range(7)}
    per_song = {}
    for n, row in enumerate(rows):
        if n >= max_rows:
            break
        c = VECTOR_TO_COMBINATION[_labels(row)]
        slot = len(picked[c])
        if slot < per_combo and row["source_category"] == CATS[(c + slot) % 3] \
                and per_song.get(row["source_song"], 0) < max_per_song:
            picked[c].append(row)
            per_song[row["source_song"]] = per_song.get(row["source_song"], 0) + 1
        elif len(backup[c]) < per_combo:
            backup[c].append(row)
        if all(len(v) == per_combo for v in picked.values()):
            break
    for c in range(7):                               # fallback if the preferred spread was not found
        for row in backup[c]:
            if len(picked[c]) < per_combo and row not in picked[c]:
                picked[c].append(row)
    return [r for c in range(7) for r in picked[c]]


def _write_pair(i, row):
    base = os.path.join(DEMO_DIR, f"pair{i:02d}")
    files = {}
    for kind in ("dry", "wet"):
        val = row[f"{kind}_audio"]
        ext = os.path.splitext(val.get("path") or "")[1] or ".wav"
        raw = f"{base}_{kind}{ext}"                  # original bytes -> model input (exactly as in training)
        with open(raw, "wb") as f:
            f.write(val["bytes"])
        files[kind] = raw
    d, _ = sf.read(files["dry"], dtype="float32", always_2d=True)
    w, sr = sf.read(files["wet"], dtype="float32", always_2d=True)
    scale = 0.99 / max(0.99, float(np.abs(d).max()), float(np.abs(w).max()))   # same gain for both: keeps level difference
    for kind, arr in (("dry", d), ("wet", w)):
        play = f"{base}_{kind}_play.wav"             # 16-bit copy for the browser player
        sf.write(play, arr.mean(axis=1) * scale, sr, subtype="PCM_16")
        files[f"{kind}_play"] = play
    keep = ["sample_id", "source_category", "source_song", "combination", "eq_params", "compressor_params",
            "reverb_params", "effect_change_db", "low_effect_change"]
    meta = {k: row.get(k) for k in keep}
    meta.update(files, labels=list(_labels(row)), combination=COMBINATIONS[VECTOR_TO_COMBINATION[_labels(row)]][0],
                pair=i, display=f"Pair {i:02d} · {row['source_category']} · {row['source_song']}")
    return meta


if os.path.exists(DEMO_JSON) and not REBUILD_DEMO:
    DEMO = json.load(open(DEMO_JSON))
    print(f"Loaded {len(DEMO)} cached demo pairs from {DEMO_JSON}")
else:
    from datasets import load_dataset, Audio
    t0 = time.time()
    stream = load_dataset(DATASET_ID, split="test", streaming=True, revision=DATASET_REVISION,
                          token=os.environ.get("HF_TOKEN"))
    stream = stream.cast_column("dry_audio", Audio(decode=False)).cast_column("wet_audio", Audio(decode=False))
    rows = select_rows(stream)
    DEMO = [_write_pair(i + 1, r) for i, r in enumerate(rows)]
    json.dump(DEMO, open(DEMO_JSON, "w"), indent=1)
    print(f"Selected and saved {len(DEMO)} test pairs in {time.time() - t0:.0f}s")

display(pd.DataFrame([{"pair": d["pair"], "category": d["source_category"], "song": d["source_song"],
                       "true combination": d["combination"]} for d in DEMO]).set_index("pair"))

## 7. Verify the app against the Kaggle test run
The app's probabilities must match `results/test_predictions.csv` (produced during training). A match proves the preprocessing here is identical to training. Small differences (≈1e-3) are normal on CPU (fp32) vs Kaggle GPU (fp16).

In [ ]:
ref = pd.read_csv(os.path.join(MODEL_DIR, "results", "test_predictions.csv")).set_index("sample_id")
RESULTS = {}
rows_v = []
for d in DEMO:
    res = predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
    RESULTS[d["pair"]] = res
    r = ref.loc[d["sample_id"]] if d["sample_id"] in ref.index else None
    diff = max(abs(res["probs"][k] - r[f"prob_{e}"]) for k, e in enumerate(EFFECTS)) if r is not None else np.nan
    flips = sum(int(res["pred"][k]) != int(r[f"pred_{e}"]) for k, e in enumerate(EFFECTS)) if r is not None else np.nan
    rows_v.append({"pair": d["pair"], "sample_id": d["sample_id"], "max |Δprob|": diff, "decision flips": flips})
ver = pd.DataFrame(rows_v).set_index("pair")
display(ver)
max_diff = float(np.nanmax(ver["max |Δprob|"])) if ver["max |Δprob|"].notna().any() else float("nan")
if np.isnan(max_diff):
    print("⚠️ None of the demo pairs were found in test_predictions.csv — cannot verify.")
elif max_diff < 0.02 and int(np.nansum(ver["decision flips"])) == 0:
    print(f"✅ VERIFIED: app matches the Kaggle evaluation (max difference {max_diff:.2e}).")
else:
    print(f"❌ MISMATCH: max difference {max_diff:.3f}. The app is not processing audio like training — do not trust it.")

## 8. Launch the Gradio app
Open the **`https://….gradio.live`** link printed below.

In [ ]:
CHOICES = [(d["display"], d["pair"]) for d in DEMO]
BY_PAIR = {d["pair"]: d for d in DEMO}


def on_select(pair):
    d = BY_PAIR[int(pair)]
    fig = spectrogram_figure(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]), d["display"])
    return d["dry_play"], d["wet_play"], fig, None, "_Press **Run model** to see the prediction and the actual effects._"


def on_run(pair):
    d = BY_PAIR[int(pair)]
    res = RESULTS.get(d["pair"]) or predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
    return probs_for_label(res), result_markdown(res, d)


def on_run_all():
    rows, ok = [], np.zeros(3)
    exact = 0
    for d in DEMO:
        res = RESULTS.get(d["pair"]) or predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
        hit = np.array([int(res["pred"][k]) == d["labels"][k] for k in range(3)])
        ok += hit
        exact += int(hit.all())
        rows.append({"Pair": d["pair"], "Category": d["source_category"], "True": d["combination"],
                     "Predicted": res["combo"], **{f"p({e})": round(float(res["probs"][k]), 3) for k, e in enumerate(EFFECTS)},
                     "Exact": "✅" if hit.all() else "❌"})
    n = len(DEMO)
    summary = (f"**{n} test pairs** · exact match **{exact}/{n} ({exact / n:.0%})** · "
               + " · ".join(f"{e} correct {int(ok[k])}/{n}" for k, e in enumerate(EFFECTS)))
    return pd.DataFrame(rows), summary


def on_upload(dry_path, wet_path):
    if not dry_path or not wet_path:
        return "Please upload **both** a dry and a wet file.", None, None, ""
    notes = []
    for name, p in (("Dry", dry_path), ("Wet", wet_path)):
        try:
            info = sf.info(p); sr, ch, dur = info.samplerate, info.channels, info.duration
        except Exception:
            y, sr = librosa.load(p, sr=None, mono=False); ch = 1 if y.ndim == 1 else y.shape[0]; dur = y.shape[-1] / sr
        if sr != SR: notes.append(f"{name}: {sr} Hz → resampled to {SR} Hz")
        if ch > 1: notes.append(f"{name}: {ch} channels → mixed to mono")
        if dur > CFG["AUDIO_LENGTH_SECONDS"] + 0.01: notes.append(f"{name}: {dur:.1f} s → only the first {CFG['AUDIO_LENGTH_SECONDS']:.0f} s is used")
        if dur < CFG["AUDIO_LENGTH_SECONDS"] - 0.01: notes.append(f"{name}: {dur:.1f} s → zero-padded to {CFG['AUDIO_LENGTH_SECONDS']:.0f} s")
        if name == "Dry": dry_dur = dur
        else: wet_dur = dur
    if abs(dry_dur - wet_dur) > 0.05:
        notes.append(f"Dry ({dry_dur:.2f} s) and wet ({wet_dur:.2f} s) differ in length — they must be the same, time-aligned recording")
    dry, wet = load_file_for_model(dry_path), load_file_for_model(wet_path)
    res = predict(dry, wet)
    warn = ("**Notes:**\n" + "\n".join(f"- {x}" for x in notes) + "\n\n" if notes else "") + \
           "_The model was trained on Pedalboard EQ/Compressor/Reverb applied to 10 s MUSDB-style stems; " \
           "other audio or other plugins may give unreliable results._"
    return warn, spectrogram_figure(dry, wet, "Uploaded pair"), probs_for_label(res), result_markdown(res)


with gr.Blocks(title="Inverse FX Predictor — Phase 1") as demo:
    gr.Markdown(f"## Inverse FX Predictor — which effects were applied?\n"
                f"Model `{MODEL_REPO}` (Log-Mel + Transformer, best epoch {state['best_epoch']}). "
                f"Inputs are only the dry and wet audio; an effect is *present* if its probability ≥ {THRESHOLD}.")
    with gr.Tabs():
        with gr.Tab("Test pairs"):
            with gr.Row():
                pick = gr.Dropdown(choices=CHOICES, value=CHOICES[0][1], label="Test pair (true answer hidden)", scale=3)
                run_btn = gr.Button("Run model", variant="primary", scale=1)
            with gr.Row():
                dry_player = gr.Audio(label="Dry (input)", type="filepath", interactive=False)
                wet_player = gr.Audio(label="Wet (processed)", type="filepath", interactive=False)
            spec_plot = gr.Plot(label="Log-mel spectrograms")
            with gr.Row():
                with gr.Column(scale=1):
                    prob_label = gr.Label(label="Effect probabilities", num_top_classes=3)
                with gr.Column(scale=2):
                    result_md = gr.Markdown()
            with gr.Accordion("Run all demo pairs", open=False):
                all_btn = gr.Button("Run all")
                all_summary = gr.Markdown()
                all_table = gr.Dataframe(interactive=False, wrap=True)
        with gr.Tab("Upload your own"):
            with gr.Row():
                up_dry = gr.Audio(label="Dry audio", type="filepath")
                up_wet = gr.Audio(label="Wet audio", type="filepath")
            up_btn = gr.Button("Predict effects", variant="primary")
            up_notes = gr.Markdown()
            up_plot = gr.Plot(label="Log-mel spectrograms")
            with gr.Row():
                with gr.Column(scale=1):
                    up_label = gr.Label(label="Effect probabilities", num_top_classes=3)
                with gr.Column(scale=2):
                    up_md = gr.Markdown()

    sel_out = [dry_player, wet_player, spec_plot, prob_label, result_md]
    pick.change(on_select, inputs=pick, outputs=sel_out)
    demo.load(on_select, inputs=pick, outputs=sel_out)
    run_btn.click(on_run, inputs=pick, outputs=[prob_label, result_md])
    all_btn.click(on_run_all, outputs=[all_table, all_summary])
    up_btn.click(on_upload, inputs=[up_dry, up_wet], outputs=[up_notes, up_plot, up_label, up_md])

demo.queue(default_concurrency_limit=2).launch(share=True, inline=False, allowed_paths=[WORK], show_error=True)

## 9. Stop the app

In [ ]:
demo.close()
print("Gradio app stopped.")